# `semantic.v_beat_rate_by_cut` — view

Thin view over Gold. No logic beyond shaping.

A view is its own definition, so there is no load step and no etl task to pair with this one.

In [0]:
-- THE THREE CUTS. One view, one extra column saying which cut a row belongs to, because
-- three near-identical views would be three things to maintain and explain.
-- Every one is a GROUP BY over a column dim_ticker already holds. No model changes.
CREATE OR REPLACE VIEW `index-vs-trust-pipeline`.semantic.v_beat_rate_by_cut
COMMENT 'Beat rate by management group, by sole/multi manager, and by AIC sector'
AS
WITH trusts AS (
  SELECT f.horizon_years, f.beat_index, f.total_return, f.volatility,
         f.risk_adjusted_return,
         d.management_group, d.manager_structure, d.aic_sector
  FROM `index-vs-trust-pipeline`.gold.fact_horizon_performance f
  JOIN `index-vs-trust-pipeline`.gold.dim_ticker d
    ON d.ticker_key = f.ticker_key AND d.entity_type = 'Trust'
),
cut AS (
  SELECT 'management group' AS cut_by, management_group  AS cut_value, * FROM trusts
  UNION ALL
  SELECT 'manager structure', manager_structure, * FROM trusts
  UNION ALL
  SELECT 'aic sector', aic_sector, * FROM trusts
)
SELECT cut_by,
       cut_value,
       horizon_years,
       COUNT(*)                                                   AS trusts,
       SUM(CASE WHEN beat_index THEN 1 ELSE 0 END)                AS beat_count,
       ROUND(100.0 * SUM(CASE WHEN beat_index THEN 1 ELSE 0 END)
             / COUNT(*), 1)                                       AS beat_rate_pct,
       ROUND(100 * PERCENTILE_APPROX(total_return, 0.5), 1)       AS median_return_pct,
       ROUND(100 * PERCENTILE_APPROX(volatility, 0.5), 1)         AS median_volatility_pct,
       ROUND(PERCENTILE_APPROX(risk_adjusted_return, 0.5), 2)     AS median_risk_adjusted
FROM cut
GROUP BY cut_by, cut_value, horizon_years;

## Verification

Expected: the view resolves and returns rows. Counts are in 
`
specs/04_semantic/dashboard.md
`
.

In [0]:
SELECT COUNT(*) AS rows
FROM `index-vs-trust-pipeline`.semantic.v_beat_rate_by_cut;